# Machine Failure Prediction — Exploratory Data Analysis

## Objective

This notebook performs exploratory data analysis on the AI4I 2020 Predictive Maintenance Dataset.

The analysis focuses on:

- Dataset structure and quality
- Missing values and duplicates
- Machine failure distribution
- Numerical feature statistics
- Feature distributions
- Relationships between operating parameters and machine failure
- Feature correlations
- Outlier analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)

In [ ]:
DATA_PATH = "../data/ai4i2020.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

display(df.head())

In [ ]:
print("Column Names:")
print(df.columns.tolist())

print("\nData Types:")
display(df.dtypes)

print("\nDataset Information:")
df.info()

In [ ]:
print("Missing Values:")
display(df.isnull().sum())

print("Total Missing Values:", df.isnull().sum().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

In [ ]:
display(df.describe())

In [ ]:
failure_counts = df["Machine failure"].value_counts()

print("Machine Failure Counts:")
display(failure_counts)

print("\nMachine Failure Percentage:")
display(
    df["Machine failure"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    data=df,
    x="Machine failure"
)

plt.title("Machine Failure Distribution")
plt.xlabel("Machine Failure")
plt.ylabel("Number of Records")
plt.xticks([0, 1], ["Normal", "Failure"])

plt.show()

### Observation

The dataset contains substantially more normal operating records than failure records. This indicates class imbalance, which should be considered during model development and evaluation.

The dataset contains 9,661 normal records and 339 failure records, corresponding to 96.61% normal operation and 3.39% machine failure.

In [ ]:
numeric_features = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

df[numeric_features].hist(
    figsize=(12, 8),
    bins=30
)

plt.suptitle("Distribution of Machine Operating Parameters")
plt.tight_layout()
plt.show()

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(7, 5))

    sns.boxplot(
        data=df,
        x="Machine failure",
        y=feature
    )

    plt.title(f"{feature} vs Machine Failure")
    plt.xlabel("Machine Failure")
    plt.ylabel(feature)
    plt.xticks([0, 1], ["Normal", "Failure"])

    plt.tight_layout()
    plt.show()

In [ ]:
correlation_data = df[
    [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]",
        "Machine failure"
    ]
]

correlation_matrix = correlation_data.corr()

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(10, 7))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm"
)

plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
failure_by_type = (
    df.groupby("Type")["Machine failure"]
    .agg(["count", "sum", "mean"])
)

failure_by_type["failure_percentage"] = (
    failure_by_type["mean"] * 100
)

display(failure_by_type)

In [ ]:
plt.figure(figsize=(6, 4))

sns.barplot(
    data=df,
    x="Type",
    y="Machine failure"
)

plt.title("Average Failure Rate by Product Type")
plt.xlabel("Product Type")
plt.ylabel("Failure Rate")

plt.show()

In [ ]:
failure_modes = [
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

failure_mode_counts = df[failure_modes].sum().sort_values(
    ascending=False
)

display(failure_mode_counts)

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(
    x=failure_mode_counts.index,
    y=failure_mode_counts.values
)

plt.title("Machine Failure Modes")
plt.xlabel("Failure Mode")
plt.ylabel("Number of Records")

plt.tight_layout()
plt.show()

In [ ]:
for feature in numeric_features:
    Q1 = df[feature].quantile(0.25)
    Q3 = df[feature].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[feature] < lower_bound) |
        (df[feature] > upper_bound)
    ]

    print(
        f"{feature}: {len(outliers)} potential outliers"
    )

# EDA Summary

The exploratory analysis produced the following observations:

1. The dataset contains 10,000 records and 14 variables.
2. No missing values were found.
3. No duplicate records were found.
4. Machine failure is an imbalanced target, with 3.39% failure records and 96.61% normal records.
5. Machine operating parameters include air temperature, process temperature, rotational speed, torque, and tool wear.
6. Boxplots were used to compare operating parameters between normal and failure conditions.
7. Correlation analysis was performed to examine relationships among numerical variables and machine failure.
8. Product-type failure rates were examined.
9. The available failure-mode variables TWF, HDF, PWF, OSF, and RNF were analyzed.
10. Potential numerical outliers were investigated using the IQR method.

These findings were used to support feature selection and model development.